# Topic: Dataset Preparation for Instruction Tuning

## Definition (30-second explanation)
Think of a base LLM as a highly-read but socially awkward librarian who just blurts out related facts when you speak. Dataset preparation for instruction tuning is like giving this librarian a script and etiquette lessons. We format raw data into a structured conversational pattern (User, Assistant, System) using special hidden tags, teaching the model not just what to say, but *how* to converse and follow instructions.

## Why Interviewers Ask This
Interviewers know that "garbage in, garbage out" is magnified in GenAI. If a candidate doesn't understand how chat templates (like ShareGPT vs. Alpaca) work, their fine-tuned model will likely output gibberish, fail to stop generating, or completely ignore system prompts. It tests whether you understand the exact pipeline between raw data and model inference.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Base models are pure next-token predictors. If you prompt them with a question, they might just generate more questions instead of an answer. The bottleneck is *alignment*—getting the model to map its vast knowledge to a specific Q&A/Chat behavior.
*   **The Mechanism:** We wrap training data in specialized tokens (e.g., `<|im_start|>user\nHello<|im_end|>`). The Hugging Face tokenizer uses a `chat_template` (a Jinja script) to automatically convert human-readable JSON lists of messages into this exact string of special tokens before feeding it to the model.
*   **The Trade-off:** Using synthetic data (like distilling from GPT-4) is cheap and fast, but heavily reliance on it can cause "mode collapse" or tone homogenization (the model sounds like a generic robot and loses edge-case reasoning).

## When to Use
*   Adapting a base model (e.g., Llama-3-8B-Base) for a specific downstream conversational task (e.g., customer support chatbot, SQL generator).
*   Aligning an already fine-tuned model to a highly specialized enterprise system prompt (e.g., forcing a strictly JSON output format).

## Advantages
*   Drastically improves the model's zero-shot performance on tasks.
*   Allows you to define a persona and strict boundary rules via the System prompt during training.
*   Standardized formats (like ShareGPT) allow you to use optimized libraries like Axolotl or Unsloth directly without writing custom data loaders.

## Limitations
*   Requires extremely high-quality data; even a few hundred bad examples can ruin the model's tone.
*   Template mismatch: If the format used during training differs *even slightly* (e.g., missing a newline character) from the format used in production inference, performance collapses.

## Common Comparisons
*   **Alpaca Format:** Best for single-turn instruction following. (Keys: `instruction`, `input`, `output`). Simple, but struggles with ongoing conversations.
*   **ShareGPT Format:** Best for multi-turn conversations. Organizes data as a list of `conversations` with `from` (human/gpt) and `value` keys. The industry standard for chat models today.

## Common Interview Traps
*   **Forgetting the EOS Token:** Candidates often forget to mention that the model must be explicitly trained to output an End-Of-Sequence (EOS) token at the end of the assistant's turn, otherwise it will just keep generating text indefinitely.
*   **Training on System/User Prompts:** The loss calculation should *only* happen on the Assistant's responses. A common trap is failing to mask the User and System prompts during the loss calculation in the training loop.

## Python / SQL Syntax (if applicable)
```python
from transformers import AutoTokenizer

# 1. Load tokenizer (which contains the Jinja chat_template)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")

# 2. Define your raw data in the standard OpenAI/ShareGPT-style format
chat = [
    {"role": "system", "content": "You are a helpful SQL assistant."},
    {"role": "user", "content": "How do I group by department?"},
    {"role": "assistant", "content": "Use the GROUP BY clause..."}
]

# 3. Apply the chat template
# tokenize=False returns the raw string with special tokens inserted.
# add_generation_prompt=True adds the final <|im_start|>assistant ready for the model to predict.
formatted_prompt = tokenizer.apply_chat_template(
    chat, 
    tokenize=False, 
    add_generation_prompt=True 
)

print(formatted_prompt)
# Output will look something like:
# <|system|>
# You are a helpful SQL assistant.</s>
# <|user|>
# How do I group by department?</s>
# <|assistant|>
```

## 45-Second Interview Answer
"Dataset preparation for instruction tuning is the process of formatting raw text into structured conversations using specialized tokens. The goal is to teach a base next-token predictor to act as a conversational agent. I typically structure my datasets using the ShareGPT format for multi-turn dialogues. In the pipeline, I use Hugging Face's apply_chat_template to inject model-specific tokens like start, end, and role identifiers. The most critical step is ensuring the loss is only calculated on the assistant's output by masking the user and system prompts, and absolutely ensuring the chat template used during fine-tuning perfectly matches the one used during inference to prevent hallucination."

## Practice Questions:

### Q1: Template Mismatch in Production
**Question:** You instruct-tuned a Llama-3 model using ShareGPT formatting. The backend team tests the deployed model by passing raw user text (e.g., `model.generate("Write a SQL query...")`) without any formatting. What happens, and how do you fix the handoff?

**Answer:**
1. **The Behavior:** The model will likely fail to answer the question. Because it doesn't see the special conversational tokens (like `<|im_start|>user`) it saw during fine-tuning, it might revert to base-model behavior (autocompleting the sentence, e.g., generating "...to find all users in Mehsana using PostgreSQL"). It might also hallucinate fake user/assistant tags, or fail to stop generating because it never outputs the End-Of-Sequence (EOS) token.
2. **The Handoff Architecture:** I would not let the backend team pass raw strings. I would provide them with a preprocessing function using Hugging Face's `tokenizer.apply_chat_template()`. This ensures the raw API input is wrapped in the exact same System/User/Assistant special tokens used during training, and crucially, I would set `add_generation_prompt=True` so the model gets the trigger token telling it that it is the assistant's turn to generate.

**Interview Tips:**
*   **The Trap:** Interviewers ask this to see if you understand that fine-tuned models are incredibly brittle to prompt formatting.
*   **Keywords to hit:** `apply_chat_template`, Base-model fallback, Special tokens, `add_generation_prompt`.

### Q2: Token Masking / Training on Prompts
**Question:** During instruction fine-tuning, why is it critical to mask the System and User prompts in the loss calculation? What happens if you don't?

**Answer:**
1. **The Core Reason:** Just like in traditional ML where we don't calculate loss on the input features, in LLM fine-tuning, we only want the model to learn how to generate the *Assistant's* response. 
2. **The Failure Mode:** Since base LLMs are next-token predictors, if we don't mask the prompts, we are actively training the model to predict what the system prompt should be, or what the user is going to ask. This wastes compute and severely degrades the model's ability to act as an assistant.
3. **The Implementation:** In PyTorch and Hugging Face, we achieve this by setting the target labels for the System and User tokens to `-100`. The Cross-Entropy Loss function in PyTorch is hardcoded to completely ignore any label with an ID of `-100`.

**Interview Tips:**
*   Dropping the `-100` detail is a massive "green flag" for interviewers. It separates people who just read concepts from people who have actually run training loops or used libraries like Axolotl/TRL.

### Q3: Format Data for Instruction Tuning
**Question:** Write a function to convert a raw Pandas DataFrame (with columns: `instruction`, `user_query`, `sql_response`) into the standard Hugging Face `messages` format for instruction tuning.

In [14]:
import pandas as pd

data = {
    "instruction": [
        "Write a SQL query to fetch all orders.",
        "Calculate total sales by product category."
    ],
    "user_query": [
        "Fetch orders placed in 2026.",
        "Show category and sum of sales, grouped by category."
    ],
    "sql_response": [
        "SELECT * FROM orders WHERE strftime('%Y', order_date) = '2026';",
        "SELECT category, SUM(sales) FROM transactions GROUP BY category;"
    ]
}

df = pd.DataFrame(data)

In [15]:
df

,instruction,user_query,sql_response
0,Write a SQL query to fetch all orders.,Fetch orders placed in 2026.,"SELECT * FROM orders WHERE strftime('%Y', orde..."
1,Calculate total sales by product category.,"Show category and sum of sales, grouped by cat...","SELECT category, SUM(sales) FROM transactions ..."


In [16]:
def format_for_hf(df, system_prompt="You are an expert SQL assistant. Output only valid SQL."):
    # to_dict('records') is much faster than iterrows() for large datasets
    raw_records = df.to_dict('records')
    
    formatted_data = []
    for row in raw_records:
        formatted_data.append({
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"{row['instruction']}\n{row['user_query']}"},
                {"role": "assistant", "content": row['sql_response']}
            ]
        })
        
    return formatted_data


In [17]:
format_for_hf(df= df)

[{'messages': [{'role': 'system',
    'content': 'You are an expert SQL assistant. Output only valid SQL.'},
   {'role': 'user',
    'content': 'Write a SQL query to fetch all orders.\nFetch orders placed in 2026.'},
   {'role': 'assistant',
    'content': "SELECT * FROM orders WHERE strftime('%Y', order_date) = '2026';"}]},
 {'messages': [{'role': 'system',
    'content': 'You are an expert SQL assistant. Output only valid SQL.'},
   {'role': 'user',
    'content': 'Calculate total sales by product category.\nShow category and sum of sales, grouped by category.'},
   {'role': 'assistant',
    'content': 'SELECT category, SUM(sales) FROM transactions GROUP BY category;'}]}]

**Interview Tips:**

- The iterrows Trap: Never use iterrows() in an interview unless you absolutely have to. Use to_dict('records'), vectorization, or standard Python zip() on columns to show you care about data pipeline performance.

- The Handoff: Mentioning how you would load this into datasets.Dataset shows end-to-end pipeline knowledge.